In [1]:
# ---------------- Imports ----------------
import json
from pathlib import Path
import os
import yaml


In [2]:
# ---------------- Args ----------------
dataset_name = "yield-v1-shuffled-finetuning"

word_length_cutoff = 2


In [3]:
# ---------------- Config ----------------
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

proj_store = config["paths"]["proj_store"]
data_path = os.path.join(proj_store, "data")
models_folderpath = config["paths"]["models"]

root_dir = os.path.join(data_path, dataset_name)
output_path = os.path.join(proj_store, "output", "utterance-analysis")
os.makedirs(output_path, exist_ok=True)
output_file = os.path.join(output_path, f"{dataset_name}-filtered-min{word_length_cutoff+1}.jsonl")
filtered_root = os.path.join(data_path, f"{dataset_name}-min{word_length_cutoff+1}")
os.makedirs(filtered_root, exist_ok=True)






In [4]:
# ---------------- Main ----------------

with open(output_file, "w", encoding="utf-8") as outfile:
    for split in ["train", "dev", "test"]:
        split_dir = os.path.join(root_dir, split)
        if not os.path.exists(split_dir):
            continue

        for root, _, files in os.walk(split_dir):
            for fname in files:
                if not fname.endswith(".jsonl"):
                    continue

                jsonl_file = os.path.join(root, fname)

                with open(jsonl_file, "r", encoding="utf-8") as infile:
                    for line in infile:
                        line = line.strip()
                        if not line:
                            continue

                        record = json.loads(line)
                        messages = record.get("messages", [])
                        if not messages:
                            continue

                        last_msg = messages[-1]
                        if last_msg.get("role") != "assistant":
                            continue

                        content = last_msg.get("content", "")
                        if len(content.split()) <= word_length_cutoff:
                            outfile.write(line + "\n")


In [5]:
# ---------------- Main ----------------



for split in ["train", "dev", "test"]:
    split_dir = os.path.join(root_dir, split)
    if not os.path.exists(split_dir):
        continue

    out_split_dir = os.path.join(filtered_root, split)
    os.makedirs(out_split_dir, exist_ok=True)

    for root, _, files in os.walk(split_dir):
        rel_root = os.path.relpath(root, split_dir)
        out_root = os.path.join(out_split_dir, rel_root)
        os.makedirs(out_root, exist_ok=True)

        for fname in files:
            if not fname.endswith(".jsonl"):
                continue

            in_file = os.path.join(root, fname)
            out_file = os.path.join(out_root, fname)

            with open(in_file, "r", encoding="utf-8") as infile, \
                 open(out_file, "w", encoding="utf-8") as outfile:

                for line in infile:
                    line = line.strip()
                    if not line:
                        continue

                    record = json.loads(line)
                    messages = record.get("messages", [])
                    if not messages:
                        outfile.write(line + "\n")
                        continue

                    last_msg = messages[-1]
                    if (
                        last_msg.get("role") == "assistant"
                        and len(last_msg.get("content", "").split()) <= word_length_cutoff
                    ):
                        continue  # DROP this record

                    outfile.write(line + "\n")
